## Assignment 3: $k$ Nearest Neighbor


**Q1.**
1. What is the difference between regression and classification?
2. What is a confusion table? What does it help us understand about a model's performance?
3. What does the SSE quantify about a particular model?
4. What are overfitting and underfitting? 
5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

**Q2.** This is a case study on $k$ nearest neighbor classification, using the `land_mines.csv` data.

The data consists of a label, `mine_type`, taking integer values 1 to 5, and three properties of the mine, `voltage`, `height` and `soil`. We want to predict the kind of mine from data about it. Imagine working for the DOD or a humanitarian aid agency, trying to help people remove land mines more safely.

1. Load the data. Perform some EDA, summarizing the target label and the features.
2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Build a $k$-NN classifier. Explain how you select $k$.
4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?
5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** [Alex Tan]

**Date:** [10/02/2026]

In [ ]:
# Your Phase 1 solution to the problem goes here.

**Q1.**
1. What is the difference between regression and classification?

Regression predicts a numeric outcome, whereas classification predicts a category.

2. What is a confusion table? What does it help us understand about a model's performance?

It creates a table to compare the model's predicted vs the actual outcomes. It allows us to see where the model is making its errors.

3. What does the SSE quantify about a particular model?

SSE measures the total prediction error. The smaller the SSE, the more accurate the predictions are to the actual values/smaller errors in the validation set.

4. What are overfitting and underfitting?

Overfitting is when the model is too focused on training that it performs poorly on new data. Underfitting is when the model is not given enough training to recognize patterns in the data.

5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?

Splitting the data allows us to see how well the model works with new data under different k values. This allows us to choose an optimal k value improve outcome/performance.

6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

Using a class label allows for easy comprehension, but lacks in amount of information returned. A probability distribution gives plenty of information, but it is more complicated to interpret.




In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (confusion_matrix, accuracy_score, classification_report)

# 1. Load the dataset
df = pd.read_csv('./data/land_mines.csv')

# EDA
print(df.head())
print(df.shape)
print(df.describe())
print(df['mine_type'].value_counts().sort_index())

# 2. Split the sample 50/50 into training and test/validation sets.
X = df[['voltage', 'height', 'soil']]
y = df['mine_type']

# Split the data 50/50
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, random_state=42, stratify=y
)

feature_scaler = MinMaxScaler()

X_train = pd.DataFrame(
feature_scaler.fit_transform(X_train),
columns=X_train.columns,
index=X_train.index
)

X_test = pd.DataFrame(
feature_scaler.transform(X_test),
columns=X_test.columns,
index=X_test.index
)

# 3. Build a k-NN classifier. Explain how you select k.
eval_ks = np.arange(1, 51)
eval_valid_sse = []
eval_train_sse = []
for eval_k in eval_ks:
    eval_candidate = KNeighborsClassifier(n_neighbors=int(eval_k))
    eval_candidate.fit(X_train, y_train)
    eval_valid_hat = eval_candidate.predict(X_test)
    eval_train_hat = eval_candidate.predict(X_train)
    eval_valid_sse.append(np.sum(
        (y_test.to_numpy() - eval_valid_hat) ** 2
    ))
    eval_train_sse.append(np.sum(
        (y_train.to_numpy() - eval_train_hat) ** 2
    ))
eval_k_star = int(eval_ks[np.argmin(eval_valid_sse)])
print("\nBest k-value:", eval_k_star, "\n")
print("Best valid_SSE:", min(eval_valid_sse), "\n")
print("\nk = 2 had the smallest validation SSE, \n"
      "making it the best k value to use. \n")


knn = KNeighborsClassifier(n_neighbors=eval_k_star)
knn.fit(X_train, y_train)
predictions = knn.predict(X_test)


# 4. Confusion table
cm = confusion_matrix(
    y_test,
    predictions
)
print("\nConfusion Table:")
print(cm)

# Overall accuracy
print("\nOverall Accuracy:")
print(accuracy_score(y_test, predictions))

print("\nBased on the results, the model is accurate at predicting \n"
      "type 2 mines, but poorly predicts other mine types, often \n"
      "mistaking them for type 1 mine. \n \n")

# 5 Notice that you can have a lot of accurate predictions for 
# a given type of mine, but still make a lot of mistakes. Please 
# explain how you'd advise someone to actually use this predictive 
# model in practice, given the errors that it tends to make.
print("This model achieves a 45.56% overall accuracy, but accuracy \n" \
      "accross individual mines differ. The model is most accurate \n" \
      "at identifying type 2 mines with a 92% accuracy. Performance \n" \
      "in other mine types drop significantly to 72%, 30%, 24%, and \n"
      "3% respectively, most of which is mistakening other type \n" \
      "mines to type 1 mine. This model can only be relied on for \n" \
      "predicting type 2 mines. Also verify the information before \n" \
      "using since predictions are not always right.")


    voltage    height  soil  mine_type
0  0.338157  0.000000   0.0          1
1  0.320241  0.181818   0.0          1
2  0.287009  0.272727   0.0          1
3  0.256284  0.454545   0.0          1
4  0.262840  0.545455   0.0          1
(338, 4)
          voltage      height        soil   mine_type
count  338.000000  338.000000  338.000000  338.000000
mean     0.430634    0.508876    0.503550    2.952663
std      0.195819    0.306043    0.344244    1.419703
min      0.197734    0.000000    0.000000    1.000000
25%      0.309737    0.272727    0.200000    2.000000
50%      0.359516    0.545455    0.600000    3.000000
75%      0.482628    0.727273    0.800000    4.000000
max      0.999999    1.000000    1.000000    5.000000
mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64

Best k-value: 2 

Best valid_SSE: 497 


k = 2 had the smallest validation SSE, 
making it the best k value to use. 


Confusion Table:
[[26  0  6  3  1]
 [ 0 32  0  3  0]
 [12  1 10  6  4]
 [ 9

## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** [To be provided]
- **AI tool and model used:** [To be provided]

### *Prompts used

1. [Paste prompt]
2. [Paste follow-up prompt]

### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. ...
2. ...

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Accept / modify / reject | ... |
| 2 | Accept / modify / reject | ... |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [ ]:
# Your Phase 2 solution to the problem goes here.


### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]